
# 🔄 Prefect Learning Project — Complete Notes
> **Source:** prefect-learning-project files only
> **Goal:** Understand Prefect from scratch — flows, tasks, retries, caching, parallel execution, ETL, deployments

---

# 1. 📖 Overview

This is a hands-on Prefect learning project structured in 3 levels — Foundation → Intermediate → Advanced. It teaches you how to build, schedule, monitor and deploy data pipelines using Prefect in Python.

**What Prefect does in simple words:**
> You write Python functions. Prefect wraps them with `@flow` and `@task` decorators. From that point, Prefect automatically tracks every run, handles failures, retries failed steps, caches results, and shows everything in a visual UI — without you writing any extra monitoring code.

**The project covers 6 flow files + deployments:**

| File | What it teaches |
|---|---|
| `01_basic_flow.py` | Flows and Tasks basics |
| `02_with_retries.py` | Error handling and retries |
| `03_with_caching.py` | Task result caching |
| `04_parallel_execution.py` | Parallel task execution |
| `05_conditional_flow.py` | Conditional logic in pipelines |
| `06_etl_pipeline.py` | Complete ETL pipeline |
| `deployments/deploy_all.py` | Scheduling and deployment |

---

# 2. ⚙️ How It Works

## The Big Picture

```
You write Python code
        ↓
Add @flow and @task decorators
        ↓
Run the flow (python flows/01_basic_flow.py)
        ↓
Prefect Engine takes over:
  → Creates a Flow Run (unique ID)
  → Tracks each task
  → Handles retries if something fails
  → Stores logs and results
  → Shows everything in UI at http://127.0.0.1:4200
```

## How Data Flows Through a Pipeline

```
Extract Task  →  Validate Task  →  Transform Task  →  Load Task
    ↓                ↓                  ↓                 ↓
  raw data       clean data        processed data     saved output
```

## How Prefect Fits In

```
Your Code (Python functions)
         +
Prefect Decorators (@flow, @task)
         =
Managed Pipeline with:
  ✅ Auto tracking
  ✅ Retry on failure
  ✅ Cached results
  ✅ Parallel execution
  ✅ Visual UI dashboard
  ✅ Scheduling (via Deployments)
```

## Server, Worker, Flow — How They Connect

```
┌─────────────────────────────────────────────┐
│           Prefect Server (Terminal 1)        │
│   → Stores metadata, runs, logs             │
│   → Hosts UI at http://127.0.0.1:4200       │
└──────────────────┬──────────────────────────┘
                   │
         ┌─────────▼──────────┐
         │    Work Pool        │
         │ (execution queue)  │
         └─────────┬──────────┘
                   │
         ┌─────────▼──────────┐
         │    Worker           │  (Terminal 3)
         │ (executes flows)   │
         └─────────┬──────────┘
                   │
         ┌─────────▼──────────┐
         │    Your Flow Code   │
         │ (flows/*.py files) │
         └────────────────────┘
```

---

# 3. 📂 Code Explanation — File by File

---

## 📄 `flows/01_basic_flow.py` — Flows and Tasks

**What it does:** Shows the most basic building blocks — a `@flow` calling multiple `@task` functions.

### Key Concepts:

```python
from prefect import flow, task

# @task = single unit of work
@task
def extract_data():
    # Gets raw data from somewhere
    return {"users": 100, "orders": 250}

@task
def transform_data(data):
    # Does something with the data
    return {k: v * 2 for k, v in data.items()}

@task
def load_data(data):
    # Saves/outputs the result
    print(f"Loading: {data}")

# @flow = the orchestrator — calls tasks in order
@flow(name="Basic Learning Flow")
def basic_flow():
    raw = extract_data()       # Step 1
    transformed = transform_data(raw)  # Step 2
    load_data(transformed)     # Step 3

# Run it
if __name__ == "__main__":
    basic_flow()
```

### What happens in the UI after running:
- Flow Run appears under "Flow Runs"
- Each task shown with duration + status
- Logs available per task
- Execution timeline visible

### Flow vs Task — The Key Difference:

| | `@flow` | `@task` |
|---|---|---|
| **Role** | Orchestrator — calls other things | Worker — does actual work |
| **Can call** | Tasks and other flows | Other tasks (carefully) |
| **Tracked** | Yes — as a Flow Run | Yes — as a Task Run |
| **Retries** | Yes | Yes (more common here) |

---

## 📄 `flows/02_with_retries.py` — Error Handling & Retries

**What it does:** Shows how to automatically retry failed tasks instead of crashing the whole pipeline.

```python
from prefect import flow, task
import random

# Task will retry 3 times, waiting 10 seconds between each retry
@task(retries=3, retry_delay_seconds=10)
def fetch_from_api():
    # Simulates an API that sometimes fails
    if random.random() < 0.7:   # 70% chance of failure
        raise Exception("API timeout!")
    return {"status": "success", "data": [1, 2, 3]}

@flow(name="Retry Demo Flow")
def retry_flow():
    result = fetch_from_api()
    print(f"Got result: {result}")

if __name__ == "__main__":
    retry_flow()
```

### Retry Flow Diagram:
```
Run 1 → FAILS (API timeout)
         ↓ wait 10 seconds
Run 2 → FAILS (API timeout)
         ↓ wait 10 seconds
Run 3 → FAILS (API timeout)
         ↓ wait 10 seconds
Run 4 → SUCCESS ✅  (or FAILED ❌ → flow marked as failed)
```

### Different Retry Strategies:
```python
# Simple retry
@task(retries=3)

# Retry with delay
@task(retries=3, retry_delay_seconds=30)

# Retry with exponential backoff (wait longer each time)
from prefect.tasks import exponential_backoff
@task(retries=4, retry_delay_seconds=exponential_backoff(backoff_factor=2))
# Waits: 2s → 4s → 8s → 16s
```

---

## 📄 `flows/03_with_caching.py` — Task Result Caching

**What it does:** Saves task results so if you run the same task again with the same inputs — it skips computation and returns the saved result instantly.

```python
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta

# Cache result for 5 minutes — same inputs = skip re-running
@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(minutes=5))
def expensive_computation(input_value: int):
    print("Computing... (this is slow)")
    import time
    time.sleep(3)   # simulate expensive work
    return input_value * 100

@flow(name="Caching Demo")
def caching_flow():
    # First call — computes (takes 3 seconds)
    result1 = expensive_computation(42)
    
    # Second call with same input — returns cached result instantly!
    result2 = expensive_computation(42)
    
    print(f"Result 1: {result1}")
    print(f"Result 2: {result2}")  # same as result1, but instant

if __name__ == "__main__":
    caching_flow()
```

### Cache Concepts:

| Concept | Explanation |
|---|---|
| `cache_key_fn` | Function that generates the cache key — `task_input_hash` uses the input values |
| `cache_expiration` | How long to keep the cached result |
| **Cache Hit** | Inputs match → return saved result, skip computation |
| **Cache Miss** | Inputs different or expired → run task normally |

### Cache Flow:
```
First run:   Input=42 → compute → save result → return result
Second run:  Input=42 → check cache → HIT → return saved result (instant!)
Third run:   Input=99 → check cache → MISS → compute → save → return
After 5min:  Input=42 → check cache → EXPIRED → compute again
```

---

## 📄 `flows/04_parallel_execution.py` — Parallel Tasks

**What it does:** Runs multiple tasks at the same time using `.submit()` and `.map()` instead of waiting for each to finish before starting the next.

```python
from prefect import flow, task

@task
def process_item(item: int):
    import time
    time.sleep(1)   # simulate work
    return item * 2

@flow(name="Parallel Execution Demo")
def parallel_flow():
    items = [1, 2, 3, 4, 5]

    # ❌ SLOW — Sequential (one at a time, takes 5 seconds)
    results_sequential = []
    for item in items:
        result = process_item(item)   # waits for each
        results_sequential.append(result)

    # ✅ FAST — Parallel with .map() (all at once, takes ~1 second)
    results_parallel = process_item.map(items)   # all run concurrently

    # Wait for parallel results
    final = [r.result() for r in results_parallel]
    print(f"Results: {final}")

if __name__ == "__main__":
    parallel_flow()
```

### Sequential vs Parallel:
```
Sequential:    [1] → [2] → [3] → [4] → [5]     = 5 seconds
               ▬▬▬▬  ▬▬▬▬  ▬▬▬▬  ▬▬▬▬  ▬▬▬▬

Parallel:      [1]                               = ~1 second
               [2]
               [3]
               [4]
               [5]
               (all running at the same time)
```

### `.submit()` vs `.map()`:
```python
# .submit() — submit a single task non-blocking
future = process_item.submit(42)   # returns immediately
result = future.result()           # wait here when needed

# .map() — submit same task for a list of inputs
futures = process_item.map([1, 2, 3, 4, 5])   # all submitted at once
results = [f.result() for f in futures]        # collect all results
```

---

## 📄 `flows/05_conditional_flow.py` — Conditional Logic

**What it does:** Makes the pipeline take different paths based on data or conditions — like an if/else in your pipeline.

```python
from prefect import flow, task

@task
def extract_data():
    return {"records": 15000, "type": "large"}

@task
def process_in_batches(data):
    print(f"Processing {data['records']} records in batches")
    return "batch_result"

@task
def process_directly(data):
    print(f"Processing {data['records']} records directly")
    return "direct_result"

@task
def validate_data(data):
    return data["records"] > 10000   # True = large dataset

@flow(name="Conditional Flow")
def conditional_flow():
    data = extract_data()
    is_large = validate_data(data)

    # Branch based on condition
    if is_large:
        result = process_in_batches(data)
    else:
        result = process_directly(data)

    print(f"Final result: {result}")

if __name__ == "__main__":
    conditional_flow()
```

### Conditional Flow Diagram:
```
extract_data()
      ↓
validate_data()
      ↓
   is_large?
   ┌───┴───┐
  YES      NO
   ↓        ↓
batches   direct
   └───┬───┘
       ↓
    final result
```

---

## 📄 `flows/06_etl_pipeline.py` — Complete ETL Pipeline

**What it does:** A real-world style pipeline — Extract from source → Validate → Clean → Transform → Load to output. This is the most complete example.

```python
from prefect import flow, task
import pandas as pd

@task(retries=2, retry_delay_seconds=5)
def extract(source_path: str):
    """Extract raw data from CSV file"""
    df = pd.read_csv(source_path)
    print(f"Extracted {len(df)} rows")
    return df

@task
def validate(df: pd.DataFrame):
    """Check data quality before processing"""
    assert df is not None, "Data is empty!"
    assert "id" in df.columns, "Missing id column!"
    print(f"Validation passed: {len(df)} rows")
    return df

@task
def clean(df: pd.DataFrame):
    """Remove nulls, duplicates"""
    df = df.dropna()
    df = df.drop_duplicates()
    print(f"After cleaning: {len(df)} rows")
    return df

@task
def transform(df: pd.DataFrame):
    """Apply business transformations"""
    df["processed"] = True
    df["value_doubled"] = df["value"] * 2
    return df

@task
def load(df: pd.DataFrame, output_path: str):
    """Save final output"""
    df.to_csv(output_path, index=False)
    print(f"Saved {len(df)} rows to {output_path}")
    return {"rows_loaded": len(df), "path": output_path}

@flow(name="ETL Pipeline", retries=1)
def etl_pipeline(
    source: str = "data/sample_data.csv",
    output: str = "data/processed/cleaned_data.csv"
):
    raw_df      = extract(source)
    valid_df    = validate(raw_df)
    clean_df    = clean(valid_df)
    final_df    = transform(clean_df)
    result      = load(final_df, output)
    return result

if __name__ == "__main__":
    etl_pipeline()
```

### ETL Pipeline Diagram:
```
data/sample_data.csv
        ↓
   [ Extract ]   ← retries=2 (file might be temporarily locked)
        ↓
   [ Validate ]  ← assert columns exist, data not empty
        ↓
   [ Clean ]     ← dropna(), drop_duplicates()
        ↓
   [ Transform ] ← add columns, apply business logic
        ↓
   [ Load ]      ← save to data/processed/cleaned_data.csv
        ↓
   { rows_loaded: N, path: "..." }  ← return metadata
```

---

## 📄 `deployments/deploy_all.py` — Deployments

**What it does:** Takes your flows from "run manually" to "runs on a schedule automatically."

```python
from prefect import flow
from prefect.deployments import Deployment
from prefect.server.schemas.schedules import CronSchedule, IntervalSchedule
from datetime import timedelta

# Import your flows
from flows.06_etl_pipeline import etl_pipeline

# Create a deployment
deployment = Deployment.build_from_flow(
    flow=etl_pipeline,
    name="ETL Pipeline - Daily",
    schedule=CronSchedule(cron="0 9 * * *"),   # runs every day at 9am
    work_pool_name="learning-pool",             # which worker pool to use
    parameters={                                # default parameters
        "source": "data/sample_data.csv",
        "output": "data/processed/cleaned_data.csv"
    }
)

if __name__ == "__main__":
    deployment.apply()   # register with Prefect server
    print("Deployment created!")
```

### Flow vs Deployment — Key Difference:

| | Flow | Deployment |
|---|---|---|
| **How to run** | `python flows/06_etl_pipeline.py` | Prefect server triggers it |
| **Scheduling** | ❌ Manual only | ✅ Cron / interval / event-based |
| **Monitoring** | Basic | Full UI monitoring |
| **Workers needed** | ❌ No | ✅ Yes |

### Schedule Types:
```python
# Run every day at 9am
CronSchedule(cron="0 9 * * *")

# Run every 30 minutes
IntervalSchedule(interval=timedelta(minutes=30))

# Run every hour
IntervalSchedule(interval=timedelta(hours=1))
```

---

# 4. 🗺️ Diagrams

## Overall Project Flow:

```
┌─────────────────────────────────────────────────────────┐
│                 PREFECT LEARNING PROJECT                 │
│                                                         │
│  Level 1 — Foundations                                  │
│  ┌──────────┐  ┌────────────┐  ┌────────────┐          │
│  │01_basic  │→ │02_retries  │→ │03_caching  │          │
│  │Flow+Task │  │Error hand. │  │Skip reruns │          │
│  └──────────┘  └────────────┘  └────────────┘          │
│                                                         │
│  Level 2 — Intermediate                                 │
│  ┌──────────┐  ┌────────────┐  ┌────────────┐          │
│  │04_parall │→ │05_conditnl │→ │07_dq_check │          │
│  │.map() ++ │  │if/else flow│  │validation  │          │
│  └──────────┘  └────────────┘  └────────────┘          │
│                                                         │
│  Level 3 — Advanced                                     │
│  ┌──────────────────┐  ┌──────────────────────┐        │
│  │ 06_etl_pipeline  │→ │ deployments/          │        │
│  │ Full E→T→L flow  │  │ Scheduled production  │        │
│  └──────────────────┘  └──────────────────────┘        │
└─────────────────────────────────────────────────────────┘
```

## ETL Data Flow:
```
CSV File
   ↓  extract()      [retries=2]
Raw DataFrame
   ↓  validate()     [assert columns exist]
Validated DataFrame
   ↓  clean()        [dropna, dedup]
Clean DataFrame
   ↓  transform()    [add columns, business logic]
Final DataFrame
   ↓  load()
Output CSV + Metadata JSON
```

## Deployment Architecture:
```
deploy_all.py
     ↓ .apply()
Prefect Server
     ↓ schedule triggers
Work Pool (queue)
     ↓ worker polls
Worker (start_worker.sh)
     ↓ executes
Your Flow (06_etl_pipeline.py)
     ↓ results
Prefect UI (http://127.0.0.1:4200)
```

---

# 5. ⚠️ Production Tips

## Flows
- Always give flows a meaningful `name=` — it shows in the UI and makes debugging easier
- Add `description=` to every flow — future you (and teammates) will thank you
- Set `log_prints=True` on flows so all `print()` statements are captured as logs automatically
- Use parameters instead of hardcoded values — makes flows reusable across environments

## Tasks
- Put `retries=` on any task that calls an external system (API, database, file system)
- Use `retry_delay_seconds=` — never retry instantly, always wait (the external system needs time to recover)
- Keep tasks small and focused — one task = one responsibility
- Return data from tasks instead of using global variables

## Caching
- Only cache tasks that are truly expensive and deterministic (same inputs = same outputs)
- Always set `cache_expiration` — never cache forever in production
- Don't cache tasks that read from live databases or APIs (data changes!)

## Parallel Execution
- Use `.map()` for processing lists of items — huge speed improvement
- Use `.submit()` for independent tasks that don't need each other's results
- Don't parallelize tasks that depend on each other — that still needs to be sequential

## Deployments
- Always use a named Work Pool — never use the default pool in production
- Set meaningful deployment names — `etl-pipeline-daily` not `deployment-1`
- Test flows manually first, then create deployments
- Start workers before triggering deployments — flows will just queue up and wait otherwise

## Server & Infrastructure
- Keep `prefect server start` running in a dedicated terminal — if it dies, no runs execute
- In real production, use **Prefect Cloud** instead of self-hosted server (managed, reliable, free tier available)
- Use environment variables for sensitive config (DB passwords, API keys) — never hardcode

---

# 6. ❌ Common Mistakes

## In Prefect Basics
- **Forgetting `@task` on functions** — Prefect won't track them, no retries, no UI visibility
- **Calling a task inside another task** — tasks should be called from flows, not from other tasks
- **Not running `prefect server start` first** — flows run but nothing appears in UI
- **Using `@flow` on everything** — flows have overhead, use `@task` for individual work units

## In Retries
- **Setting retries too high** — `retries=10` means you wait a long time before getting an alert that something is broken
- **Not setting `retry_delay_seconds`** — retrying instantly often fails for the same reason (server still down)
- **Retrying tasks that shouldn't retry** — e.g. a task that writes to a DB shouldn't retry blindly (could duplicate data)

## In Caching
- **Caching tasks with side effects** — if a task sends an email or writes to a DB, caching it means it won't run again = email not sent
- **Not setting cache expiration** — stale cached data gets returned long after it should have been refreshed
- **Expecting cache to persist across server restarts** — local cache is lost when server restarts

## In Parallel Execution
- **Using `.map()` on tasks with dependencies** — if task B needs task A's output, they can't be parallelized
- **Not collecting `.result()`** — if you don't call `.result()`, the flow finishes before tasks complete
- **Parallelizing too aggressively** — too many parallel tasks can overwhelm databases or APIs with connections

## In ETL Pipelines
- **Not validating data before transforming** — garbage in = garbage out, and it's harder to debug later
- **Doing too much in one task** — hard to retry, hard to debug, hard to cache
- **Not returning metadata from `load()`** — you lose visibility into what was actually loaded

## In Deployments
- **Creating a deployment without a running worker** — flows queue up but never execute
- **Not connecting worker to the right work pool** — worker sits idle, deployment never triggers
- **Scheduling without testing** — always run the flow manually first to verify it works

## In Workflow Design
- **Putting business logic in the flow function** — flow should only orchestrate, logic goes in tasks
- **Not handling exceptions explicitly** — let tasks fail cleanly so Prefect can retry properly
- **Hardcoding file paths** — use parameters so flows work in dev, staging and prod without code changes

---

# 7. 🚀 Quick Command Reference

```bash
# Terminal 1 — Start Prefect server (keep running)
prefect server start

# Terminal 2 — Run flows
python flows/01_basic_flow.py
python flows/02_with_retries.py
python flows/03_with_caching.py
python flows/04_parallel_execution.py
python flows/05_conditional_flow.py
python flows/06_etl_pipeline.py

# Create deployments
python deployments/deploy_all.py

# Terminal 3 — Start worker
prefect work-pool create "learning-pool" --type process
bash scripts/start_worker.sh

# Useful Prefect CLI commands
prefect flow-run ls              # list all flow runs
prefect deployment ls            # list all deployments
prefect work-pool ls             # list work pools
```

---

# 8. 💡 One-Liner Explanations (Say These Confidently)

| Concept | One-liner |
|---|---|
| **Prefect** | Workflow orchestration tool — build, schedule and monitor data pipelines in Python |
| **Flow** | The entire workflow decorated with `@flow` — orchestrates multiple tasks |
| **Task** | A single unit of work decorated with `@task` — can retry, cache and run in parallel |
| **Deployment** | Makes a flow run on a schedule — connects flows to workers via work pools |
| **Worker** | Executes flows — polls work pools for scheduled runs |
| **Work Pool** | Execution queue — connects deployments to workers |
| **Cache** | Saves task results — same inputs skip recomputation, return saved result |
| **Future** | What `.submit()` returns — a handle to a task running in background |
| **`.map()`** | Run same task over a list of inputs in parallel |

---

## 📌 Quick Recap

```
Project structure:
  flows/         → 6 flow files (01 to 06, increasing complexity)
  deployments/   → deploy_all.py (scheduling)
  scripts/       → start_server.sh, start_worker.sh
  data/          → sample_data.csv + processed/ output folder

Learning path:
  01 basic → 02 retries → 03 caching → 04 parallel → 05 conditional → 06 ETL → deployments

Key decorators:
  @flow   → orchestrator (calls tasks, tracked as Flow Run)
  @task   → worker unit (retryable, cacheable, parallelizable)

Running locally:
  Terminal 1: prefect server start
  Terminal 2: python flows/XX_flow.py
  Terminal 3: bash scripts/start_worker.sh  (only for deployments)

UI: http://127.0.0.1:4200
```

---

*Notes built from: LEARNING_GUIDE.md, QUICK_START.md, and project file structure*